#### 1번 CIC 데이터셋 + DoS 데이터셋 

전처리 과정
(CIC-BruteForce데이터셋 + 가상 환경에서 DoS Slowloris 공격 패킷 수집 데이터셋)에 전처리 적용

drop 컬럼
- 값이 없는 고정 컬럼 10개
- 두 공격 데이터셋에서 확인된 완전 중복 컬럼 5개: Subflow Fwd Packets, Subflow Bwd Packets, Subflow Fwd Bytes, Avg Fwd Segment Size, Fwd Header Length.1
- 중요도 하위 컬럼은 삭제 후 F1이 낮아졌으므로 유지

input데이터
1. DoS.csv : 재가공안한 원본 CIC-DDos 데이터셋(수)
2. doslabeldata.csv : 가상 환경에서 DoS slowloris 공격 패킷 수집 데이터셋(cicflowmeter에서 매핑 전처리만 완료. drop X)

output 데이터
1. DoS_new.csv : (CIC-DDoS데이터셋 + 가상 환경에서 slowloris 공격 패킷 수집 데이터셋)에 전처리 적용. 

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

def load_data(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    return df

def remove_constant_columns(df):
    drop_cols = [
        "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags", "CWE Flag Count",
        "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
        "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",

        # 추가된 중복 column 제거
        "Subflow Fwd Packets", "Subflow Bwd Packets",
        "Subflow Fwd Bytes", "Avg Fwd Segment Size",
        "Fwd Header Length.1"
    ]
    drop_cols = [col for col in drop_cols if col in df.columns]
    return df.drop(columns=drop_cols)

def convert_time_columns_to_seconds(df):
    # source가 "cic_original"인 행만 마이크로초->초 변환
    time_cols = [
        "Flow Duration",
        "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
        "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
        "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
        "Active Mean", "Active Std", "Active Max", "Active Min",
        "Idle Mean", "Idle Std", "Idle Max", "Idle Min",
    ]
    time_cols = [col for col in time_cols if col in df.columns]
    df[time_cols] = df[time_cols].astype(float)
    mask = df["source"] == "cic_original"
    df.loc[mask, time_cols] = df.loc[mask, time_cols] / 1_000_000
    return df

def handle_labels(df):
    # 하나의 DoS 라벨로 통합
    dos_labels = [
        #"DoS GoldenEye",
        #"DoS Hulk",
        #"DoS Slowhttptest",
        "DoS slowloris"
    ]

    df["Label"] = df["Label"].replace(dos_labels, "DoS")

    # HeartBleed 라벨 제거
    df = df[df["Label"] != "Heartbleed"]

    # slowloris 제외 일단 삭제하는거로
    df = df[df["Label"] != "DoS GoldenEye"]
    df = df[df["Label"] != "DoS Hulk"]
    df = df[df["Label"] != "DoS Slowhttptest"]


    return df

def remove_empty_payload_dos(df):
    before = len(df)
    df = df[~((df["Label"] == "DoS") & (df["Max Packet Length"] == 0))]
    print(f"페이로드 없는 DoS {before - len(df)}행 제거")
    return df

def handle_missing(df):
    df = df.replace([np.inf, -np.inf], np.nan)
    return df.dropna()


project_dir = Path(r"C:\Users\구경섭\OneDrive\문서\study\SK-Shilders\python\Python_project\Module_Project_2")
cic_path = project_dir / "DoS.csv"
new_path = project_dir / "doslabeldata.csv"
output_path = project_dir / "DoS_new.csv"

cic_raw = load_data(cic_path)
cic_raw["source"] = "cic_original"

new_raw = load_data(new_path)
new_raw["source"] = "vmware_capture"

df = pd.concat([cic_raw, new_raw], ignore_index=True)

df = remove_constant_columns(df)
df = convert_time_columns_to_seconds(df)   # source 기준으로 분기 처리됨
df = handle_missing(df)
df = handle_labels(df)
df = remove_empty_payload_dos(df)

df = df.drop(columns=["source"])           # 학습 피처 아니므로 최종 저장 전 제거
df.to_csv(output_path, index=False)
print(df["Label"].value_counts())

페이로드 없는 DoS 1647행 제거
Label
BENIGN    439683
DoS         9852
Name: count, dtype: int64


#### 2번 CIC 데이터셋 + BruteForce 데이터셋 

전처리 과정
(CIC-BruteForce데이터셋 + 가상 환경에서 BruteForce 공격 패킷 수집 데이터셋)에 전처리 적용

drop 컬럼(1번과 동일)
- 값이 없는 고정 컬럼 10개
- 두 공격 데이터셋에서 확인된 완전 중복 컬럼 5개: Subflow Fwd Packets, Subflow Bwd Packets, Subflow Fwd Bytes, Avg Fwd Segment Size, Fwd Header Length.1
- 중요도 하위 컬럼은 삭제 후 F1이 낮아졌으므로 유지

input데이터
1. BF.csv : 재가공안한 원본 CIC-BruteForce 포함 데이터셋(목)
2. bflabeldata.csv : 가상 환경에서 BruteForce 공격 패킷 수집 데이터셋(cicflowmeter에서 매핑 전처리만 완료. drop X)

output 데이터
1. BF_new.csv : (CIC-BruteForce데이터셋 + 가상 환경에서 BruteForce 공격 패킷 수집 데이터셋)에 전처리 적용. 

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

def load_data(file_path):
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    return df

def remove_constant_columns(df):
    drop_cols = [
        "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags", "CWE Flag Count",
        "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate",
        "Bwd Avg Bytes/Bulk", "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate",

        # 추가된 중복 column 제거
        "Subflow Fwd Packets", "Subflow Bwd Packets",
        "Subflow Fwd Bytes", "Avg Fwd Segment Size",
        "Fwd Header Length.1"
    ]
    drop_cols = [col for col in drop_cols if col in df.columns]
    return df.drop(columns=drop_cols)

def convert_time_columns_to_seconds(df):
    # source가 "cic_original"인 행만 마이크로초->초 변환
    time_cols = [
        "Flow Duration",
        "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
        "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max", "Fwd IAT Min",
        "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min",
        "Active Mean", "Active Std", "Active Max", "Active Min",
        "Idle Mean", "Idle Std", "Idle Max", "Idle Min",
    ]
    time_cols = [col for col in time_cols if col in df.columns]
    df[time_cols] = df[time_cols].astype(float)
    mask = df["source"] == "cic_original"
    df.loc[mask, time_cols] = df.loc[mask, time_cols] / 1_000_000
    return df


def handle_labels(df):
    df.loc[df["Label"].str.contains("Brute Force", case=False, na=False), "Label"] = "Brute Force"
    df = df[~df["Label"].str.contains("Sql Injection", case=False, na=False)]
    df = df[~df["Label"].str.contains("XSS", case=False, na=False)]
    return df

def remove_empty_payload_bruteforce(df):
    before = len(df)
    df = df[~((df["Label"] == "Brute Force") & (df["Max Packet Length"] == 0))]
    print(f"페이로드 없는 BF {before - len(df)}행 제거")
    return df

def handle_missing(df):
    df = df.replace([np.inf, -np.inf], np.nan)
    return df.dropna()


project_dir = Path(r"C:\Users\구경섭\OneDrive\문서\study\SK-Shilders\python\Python_project\Module_Project_2")
cic_path = project_dir / "BF.csv"
new_path = project_dir / "bflabeldata.csv"
output_path = project_dir / "BF_new.csv"

cic_raw = load_data(cic_path)
cic_raw["source"] = "cic_original"

new_raw = load_data(new_path)
new_raw["source"] = "vmware_capture"

df = pd.concat([cic_raw, new_raw], ignore_index=True)

df = remove_constant_columns(df)
df = convert_time_columns_to_seconds(df)   # source 기준으로 분기 처리됨
df = handle_missing(df)
df = handle_labels(df)
df = remove_empty_payload_bruteforce(df)

df = df.drop(columns=["source"])           # 학습 피처 아니므로 최종 저장 전 제거
df.to_csv(output_path, index=False)
print(df["Label"].value_counts())

페이로드 없는 BF 1356행 제거
Label
BENIGN         168051
Brute Force      7653
Name: count, dtype: int64


3번 BF파일과 DoS 데이터셋 통합! 
하나의 머신러닝으로 구현 후 공격 분류하기 위함!!!

input데이터
1. BF_new.csv : (CIC-BruteForce데이터셋 + 가상 환경에서 BruteForce 공격 패킷 수집 데이터셋)에 전처리 적용. 전처리 코드= ML팀/전처리/합병/merge.ipynb)
2. DoS_new.csv : (CIC-DoS데이터셋 + 가상 환경에서 DoS Slowloris 공격 패킷 수집 데이터셋)에 전처리 적용. 전처리 코드= ML팀/전처리/합병/merge.ipynb)

output데이터
1. finaldataset.csv

In [6]:
import pandas as pd

bf = pd.read_csv("BF_new.csv")
bf.columns = bf.columns.str.strip()

dos = pd.read_csv("DoS_new.csv")
dos.columns = dos.columns.str.strip()

# 컬럼 구성 다른지 체크
print("BF에만 있는 컬럼:", set(bf.columns) - set(dos.columns))
print("DoS에만 있는 컬럼:", set(dos.columns) - set(bf.columns))

combined = pd.concat([bf, dos], ignore_index=True)

# BENIGN이 두 파일 모두에 들어있어서 완전히 똑같은 행이 겹칠 수도 있으니 중복 제거
before = len(combined)
combined = combined.drop_duplicates()
print(f"중복 제거: {before - len(combined)}행")

combined.to_csv("final_dataset.csv", index=False)
print(combined["Label"].value_counts())

BF에만 있는 컬럼: set()
DoS에만 있는 컬럼: set()
중복 제거: 35402행
Label
BENIGN         572751
DoS              9433
Brute Force      7653
Name: count, dtype: int64
